# MCNO: Monte Carlo-Type Neural Operator — Demo

This notebook demonstrates training and evaluating MCNO on the KdV equation benchmark.

**Paper**: [Monte Carlo-Type Neural Operator for One-Dimensional Differential Equations](https://arxiv.org/abs/2511.18930)

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.io import loadmat
from timeit import default_timer

import sys
sys.path.append("..")
from mcno.model import Net1d
from mcno.utils import LpLoss

torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Configuration

In [ ]:
ntrain = 1000
ntest = 100
sub = 2**5
s = 2**13 // sub

batch_size = 20
learning_rate = 0.001
epochs = 500
step_size = 100
gamma = 0.5
width = 64
num_samples = 150

## 2. Load Data

In [ ]:
raw = loadmat("../data/kdv_train_test.mat")
x_data = raw["input"].astype(np.float32)
y_data = raw["output"].astype(np.float32)

x_train = torch.from_numpy(x_data[:ntrain, ::sub])
y_train = torch.from_numpy(y_data[:ntrain, ::sub])
x_test = torch.from_numpy(x_data[-ntest:, ::sub])
y_test = torch.from_numpy(y_data[-ntest:, ::sub])

grid = torch.linspace(0, 1, s).reshape(1, s, 1)
x_train = torch.cat([x_train.reshape(ntrain, s, 1), grid.repeat(ntrain, 1, 1)], dim=2)
x_test = torch.cat([x_test.reshape(ntest, s, 1), grid.repeat(ntest, 1, 1)], dim=2)

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(x_train, y_train),
    batch_size=batch_size, shuffle=True,
)
test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(x_test, y_test),
    batch_size=batch_size, shuffle=False,
)

print(f"Train: {x_train.shape}, Test: {x_test.shape}")

## 3. Initialize Model

In [ ]:
model = Net1d(width, s, num_samples).to(device)
print(f"Parameters: {model.count_params():,}")

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
loss_fn = LpLoss(size_average=False)

## 4. Train

In [ ]:
train_l2_history = []
test_l2_history = []

for ep in range(epochs):
    model.train()
    t1 = default_timer()
    train_mse = 0.0
    train_l2 = 0.0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        batch = x.size(0)

        optimizer.zero_grad()
        out = model(x)
        mse = F.mse_loss(out, y, reduction="mean")
        l2 = loss_fn(out.view(batch, -1), y.view(batch, -1))
        l2.backward()
        optimizer.step()

        train_mse += mse.item()
        train_l2 += l2.item()

    scheduler.step()

    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            batch = x.size(0)
            out = model(x)
            test_l2 += loss_fn(out.view(batch, -1), y.view(batch, -1)).item()

    train_mse /= len(train_loader)
    train_l2 /= ntrain
    test_l2 /= ntest
    t2 = default_timer()

    train_l2_history.append(train_l2)
    test_l2_history.append(test_l2)

    if ep % 50 == 0 or ep == epochs - 1:
        print(f"Epoch {ep:4d} | Time {t2-t1:.2f}s | Train MSE {train_mse:.6f} | "
              f"Train L2 {train_l2:.6f} | Test L2 {test_l2:.6f}")

## 5. Training Curves

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.semilogy(train_l2_history, label="Train Relative L2")
ax.semilogy(test_l2_history, label="Test Relative L2")
ax.set_xlabel("Epoch")
ax.set_ylabel("Relative L2 Error")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title("MCNO Training — KdV Equation")
plt.tight_layout()
plt.show()

## 6. Evaluate & Visualize Predictions

In [ ]:
eval_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(x_test, y_test),
    batch_size=1, shuffle=False,
)

predictions = []
errors = []

model.eval()
with torch.no_grad():
    for x, y in eval_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        l2 = loss_fn(out.view(1, -1), y.view(1, -1)).item()
        predictions.append(out.cpu().numpy().squeeze())
        errors.append(l2)

print(f"Mean Test L2: {np.mean(errors):.6f}")
print(f"Median Test L2: {np.median(errors):.6f}")

In [ ]:
grid_pts = np.linspace(0, 1, s)
indices = [0, 25, 50, 75]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, idx in zip(axes.flat, indices):
    truth = y_test[idx].numpy()
    pred = predictions[idx]
    ax.plot(grid_pts, truth, "b-", label="Ground Truth", linewidth=1.5)
    ax.plot(grid_pts, pred, "r--", label="MCNO", linewidth=1.5)
    ax.set_title(f"Sample {idx} (L2 = {errors[idx]:.4f})")
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle("MCNO Predictions — KdV Equation", fontsize=14)
plt.tight_layout()
plt.show()